In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sgrna_subsample
import subprocess
%matplotlib inline

# from Bio import SeqIO
from Bio.Seq import Seq

# Set a constant seed for pandas sample
np.random.seed(0)

# CRISPRi essential library design

The data is from

Hawkins, John S., Melanie R. Silvis, Byoung-Mo Koo, Jason M. Peters, Hendrik Osadnik, Marco Jost, Cameron C. Hearne, Jonathan S. Weissman, Horia Todor, and Carol A. Gross. 2020. “Mismatch-CRISPRi Reveals the Co-Varying Expression-Fitness Relationships of Essential Genes in Escherichia Coli and Bacillus Subtilis.” Cell Systems 11 (5): 523–35.e9.

## Bin on fitness and predicted KD axes

This part bins according to the measured fitness and predicted KD values on the paper. It tries to makes the most even binning as posible according to the intervals defined on `bin_dict`

- Load the median traces, the points, and the MADs
- For each gene:
    - Bin the predicted knockdown value uniformly (maybe use dict)
    - For each bin, select K random gRNAs according to their proximity to the median curve and the MAD at that point. Different criteria can be used per bin
        - Select the closest ones to the median line
        - Select randomly among the ones that are within some distance, else expand until getting max per bin or 
        - Just select randomly i.e. do the same I did for fitness but for bins. I am leaning towards this strategy. Do it and look at a couple of them

In [ ]:
bin_dict = {
    0:[-0.5, 0.1], # Lower bound 
    1:[0.1, 0.2],
    2:[0.2, 0.3],
    3:[0.3, 0.4],
    4:[0.4, 0.5],
    5:[0.5, 0.6],
    6:[0.6, 0.7],
    7:[0.7, 0.8],
    8:[0.8, 0.9],
    9:[0.9, 1.5] # Cannot set upper bound bc there are fitness values >= 1
}

k = 2 # Number of gRNAs per bin

headpath = '/home/gutil/scratch/2023-11-15_Mismatch-CRISPRi-Essential-Genes/'
filename_essentials = 'Table_S3_1-s2.0-S2405471220303665-mmc4.xlsx'
filename_controls = 'Table_S3_1-s2.0-S2405471220303665-mmc4.xlsx'
filename_auxotrophs = 'KO_S4_mmc5.xlsx'

In [ ]:
# Read Excel file
# table_url = "https://ars.els-cdn.com/content/image/1-s2.0-S2405471220303665-mmc4.xlsx"
df = (pd.read_excel(headpath + filename_essentials,
                    sheet_name="relative fitness (bsu data)")
    # .loc[lambda df_: df_["family_retained"] == True] # Remove aberrant gRNAs according to the paper
    .drop(columns=["family_retained"]) # Drop column
    .rename(columns={'relative fitness (predicted)':'predicted_efficacy',
                     'relative fitness (mean)':'relative_fitness_mean',
                     'relative fitness (stddev)':'relative_fitness_std',
                     'variant': 'mismatch_sgrna',
                     'original': 'full_sgrna'}) # Rename column
    .astype({"pam": "category"}) # Set type of pam to categorical
    # .loc[lambda df_: df_["pam"] != "AGG"] # Remove AGG PAMs
    # .loc[lambda df_: df_["offset"] <= 1000] # Remove rows with offset > 1,000
    # Remove row with nans
    .dropna(subset=['relative_fitness_mean'])
    .sort_values(["gene", "relative_fitness_mean"])
    .assign(locus_tag = lambda df_: df_['locus_tag'].str.slice(0, 3)+'_'+df_['locus_tag'].str.slice(3)) # Add underscore to locus tag
    
    # Assign bin indices
    # .assign(bin_index_fitness = lambda df_:sgrna_subsample.get_bin_ids(df_,
    #                                                                 col='relative_fitness_mean',
    #                                                                 bin_dict=bin_dict))
    # .loc[lambda df_: df_["bin_index_fitness"] != -1] # Remove rows with bin_index_fitness == -1
    # .assign(bin_index = lambda df_:sgrna_subsample.get_bin_ids(df_,
    #                                                            col='predicted_efficacy',
    #                                                            bin_dict=bin_dict))
    # .loc[lambda df_: df_["bin_index"] != -1] # Remove rows with bin_index == -1

    # Select the rows whose mismatch_grna does not contain any restriction site
    # .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.check_all_sites) == False]

    # Select the rows whose sequence to be ordered does not contain BsaI or PvUI restriction sites
    # .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs).apply(sgrna_subsample.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False]
)
df

In [ ]:
# Get the last 5 characters of mismatch_grna
df = df.assign(seed_5nt = lambda df_: df_['mismatch_sgrna'].str.slice(-5))

In [ ]:
# Group by seed_5nt and gene and get the mean, the standard deviantion, and the count
df_seeds = df.groupby(['seed_5nt', 'gene']).agg({'relative_fitness_mean': ['mean', 'std', 'count']}).reset_index()

# Remove the multi-index
df_seeds.columns = ['seed_5nt', 'gene', 'mean_by_gene', 'std_by_gene', 'count_by_gene']
df_seeds = df_seeds.loc[:, ['seed_5nt', 'mean_by_gene']]
# Now group by seed_5nt
# df_seeds = df_seeds.groupby('seed_5nt').agg({'mean': ['mean'], 'std': ['mean'], 'count': ['mean']}).reset_index()

In [ ]:
df2 = df_seeds.groupby('seed_5nt').agg({'mean_by_gene': ['mean', 'std', 'count']}).reset_index()

# Make single-indexed
df2.columns = ['seed_5nt', 'mean_by_seed', 'std_by_seed', 'count_by_seed']

In [ ]:
df2.sort_values('mean_by_seed', ascending=True).head(20)

In [ ]:
df2['mean_by_seed'].hist(bins=30)

In [ ]:
# Read Excel file
# table_url = "https://ars.els-cdn.com/content/image/1-s2.0-S2405471220303665-mmc4.xlsx"
df = (pd.read_excel(headpath + filename_essentials,
                    sheet_name="relative fitness (bsu data)")
    .loc[lambda df_: df_["family_retained"] == True] # Remove aberrant gRNAs according to the paper
    .drop(columns=["family_retained"]) # Drop column
    .rename(columns={'relative fitness (predicted)':'predicted_efficacy',
                     'relative fitness (mean)':'relative_fitness_mean',
                     'relative fitness (stddev)':'relative_fitness_std',
                     'variant': 'mismatch_sgrna',
                     'original': 'full_sgrna'}) # Rename column
    .astype({"pam": "category"}) # Set type of pam to categorical
    .loc[lambda df_: df_["pam"] != "AGG"] # Remove AGG PAMs
    .loc[lambda df_: df_["offset"] <= 1000] # Remove rows with offset > 1,000
    # Remove row with nans
    .dropna(subset=['relative_fitness_mean'])
    .sort_values(["gene", "relative_fitness_mean"])
    .assign(locus_tag = lambda df_: df_['locus_tag'].str.slice(0, 3)+'_'+df_['locus_tag'].str.slice(3)) # Add underscore to locus tag
    
    # Assign bin indices
    .assign(bin_index_fitness = lambda df_:sgrna_subsample.get_bin_ids(df_,
                                                                    col='relative_fitness_mean',
                                                                    bin_dict=bin_dict))
    .loc[lambda df_: df_["bin_index_fitness"] != -1] # Remove rows with bin_index_fitness == -1
    .assign(bin_index = lambda df_:sgrna_subsample.get_bin_ids(df_,
                                                               col='predicted_efficacy',
                                                               bin_dict=bin_dict))
    .loc[lambda df_: df_["bin_index"] != -1] # Remove rows with bin_index == -1

    # Select the rows whose mismatch_grna does not contain any restriction site
    .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.check_all_sites) == False]

    # Select the rows whose sequence to be ordered does not contain BsaI or PvUI restriction sites
    .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs).apply(sgrna_subsample.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False]
)
df

In [ ]:
# Subsample the gRNAs for all genes based on the measured fitness
df_me = (df
         .groupby('gene')
         .apply(sgrna_subsample.generate_sample_single_gene,
                bin_dict=bin_dict,
                k=1, col='relative_fitness_mean',
                bin_index_name='bin_index_fitness',
                include_groups=False)
         .reset_index()
         .rename(columns={'level_1': 'index_original'})
         .assign(category = 'essential_measured')
         .astype({'category': 'category'})
         .assign(seq_to_order = lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs))
       #   .loc[lambda df_: df_['seq_to_order'].apply(check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Redundant
         
)
# df_me.to_csv(headpath+'df_me_essentials_measured.csv', index = False)

# Generate the gRNAs for all genes based on the predicted fitness
df_pr = (df.loc[~df.index.isin(df_me['index_original'])]
         .groupby('gene')
         .apply(sgrna_subsample.generate_sample_single_gene,
                bin_dict=bin_dict,
                k=1, col='predicted_efficacy',
                bin_index_name='bin_index',
                include_groups=False)
         .reset_index()
         .rename(columns={'level_1': 'index_original'})
         .assign(category = 'essential_predicted')
         .astype({'category': 'category'})
         .assign(seq_to_order = lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs))
       #   .loc[lambda df_: df_['seq_to_order'].apply(check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Redundant
)
# df_pr.to_csv('df_me_essentials_predicted.csv', index=False)

In [ ]:
df_me['gene'].value_counts()

### Include controls

In [ ]:
# Read Excel file
# table_url = "https://ars.els-cdn.com/content/image/1-s2.0-S2405471220303665-mmc4.xlsx"
df_controls = (pd.read_excel(headpath+filename_controls,
                    sheet_name="relative fitness (bsu controls)")
                 # Remove row with nans
                 .rename(columns={'relative fitness (stddev)':'relative_fitness_std',
                                 'relative fitness (mean)':'relative_fitness_mean',
                                 'variant': 'mismatch_sgrna',
                                 'original': 'full_sgrna'}) # Rename column
                 .dropna(subset=['relative_fitness_mean'])
                 .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.check_all_sites) == False] # Remove rows with restriction sites
                 .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs).apply(sgrna_subsample.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Remove rows with BsaI and PvuI sites
                 # Remove rows for which 'relative fitness (stdev)' is less than 0.02
                 .loc[lambda df_: df_['relative_fitness_std'] < 0.02]
                 # Filter rows whose relative_fitness_mean is between 1 + delta and 1 - delta
                 .loc[lambda df_: df_['relative_fitness_mean'].between(1-0.003, 1+0.003)] # TODO: Let delta be variable?
                 .sample(n=100, replace=False) # Sample 100 controls
                 .assign(category = 'control')
                 .astype({'category': 'category'})
                 .assign(seq_to_order = lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs))
                #  .loc[lambda df_: df_['seq_to_order'].apply(check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Check for BsaI and PvuI sites only
    )
# TODO: Find mismatches

In [ ]:
# df_library = (pd.concat([df_me, df_pr, df_controls])
#  .sort_values(['gene', 'relative_fitness_mean'])
#  .reset_index(drop=True)
# )
df_controls.to_csv(headpath+'df_controls.csv', index=False)

### Include auxotrophs
This part requires running their code to generate the gRNA list given the locus_tags

In [ ]:
genes_already_used = set(df_me['gene'].unique()) | set(df_pr['gene'].unique())

In [ ]:
# Read Excel file
# table_url = "https://ars.els-cdn.com/content/image/1-s2.0-S2405471220303665-mmc4.xlsx"
df_aux = (pd.read_excel(headpath+filename_auxotrophs,
                    sheet_name="D",
                    skiprows=16,
                    usecols = ['Locus tag', 'gene']
                    )
        # Add _ to the Locus tag column
        .rename(columns={"Locus tag": "locus_tag"})
        .assign(locus_tag = lambda df_: df_['locus_tag'].str.slice(0, 3)+'_'+df_['locus_tag'].str.slice(3))
        .loc[lambda df_: ~df_['gene'].isin(genes_already_used)] # Filter out genes already used
    )

In [ ]:
path_to_script = '/home/gutil/lag_paulsson-lab/library_design/generate_sgrnas.py'
path_to_genbank = headpath+'GCF_000009045.1_ASM904v1_genomic_Bsubtilis-168.gbff'
path_to_params = headpath+ 'model_param_bsub.csv'
path_to_output_find = headpath+'sgrnas_auxotrophs.csv'
path_to_output_filter = headpath+'sgrnas_auxotrophs_filtered.csv'
path_to_output_mismatch = headpath+'sgrnas_auxotrophs_mismatch.csv'

path_log = headpath+'log.txt'

In [ ]:
# Find
with open(path_log, 'w') as f:
    subprocess.run(['python', path_to_script, 
                    '--genbank', path_to_genbank,
                    '--locus_tag', ','.join(df_aux['locus_tag'].to_list()),
                    '--model_param', path_to_params, 
                    '--step', 'find',
                    '--file_find', path_to_output_find], stdout=f, stderr=f)

In [ ]:
# Filter
with open(path_log, 'w') as f:
    subprocess.run(['python', path_to_script, 
                    '--genbank', path_to_genbank,
                    '--locus_tag', ','.join(df_aux['locus_tag'].to_list()),
                    '--model_param', path_to_params, 
                    '--step', 'filter',
                    # '--sgrna_remove', ,
                    '--pam_remove', 'ANN',
                    '--gc_upper', '0.6',
                    '--off_target_upper', '0',
                    '--file_find', path_to_output_find,
                    '--file_filter', path_to_output_filter
                    ], stdout=f, stderr=f)

In [ ]:
# Mismatch
with open(path_log, 'w') as f:
    subprocess.run(['python', path_to_script, 
                    '--genbank', path_to_genbank,
                    '--locus_tag', ','.join(df_aux['locus_tag'].to_list()),
                    '--model_param', path_to_params, 
                    '--step', 'mismatch',
                    '--file_find', path_to_output_find,
                    '--file_filter', path_to_output_filter,
                    '--file_mismatch', path_to_output_mismatch
                    # '--sgrna_remove', ,
                    ], stdout=f, stderr=f)

In [ ]:
df = (pd.read_csv(path_to_output_mismatch,
                  usecols=['locus_tag', 'n', 'position', 'original', 'substitution', 'full_sgrna', 'mismatch_sgrna', 'predicted efficacy'])
      .rename(columns={'predicted efficacy': 'predicted_efficacy'})
      .merge(df_aux, on='locus_tag', how='left') # Recover gene names
      .assign(bin_index = lambda df_:sgrna_subsample.get_bin_ids(df_,
                                                                    col='predicted_efficacy',
                                                                    bin_dict=bin_dict)) # Assign bin indices to each gRNA
      .loc[lambda df_: df_.bin_index != -1] # Remove gRNAs that were not assigned to a 
      .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.check_all_sites) == False] # Remove rows with restriction sites
      .loc[lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs).apply(sgrna_subsample.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Remove rows with BsaI and PvuI sites                                                                                          
)

In [ ]:
df_grna_aux = (df
              .groupby('gene')
              .apply(sgrna_subsample.generate_sample_single_gene,
                     bin_dict=bin_dict,
                     k=2, 
                     col='predicted_efficacy',
                     bin_index_name='bin_index',
                     include_groups=False)
              .reset_index()
              .rename(columns={'level_1': 'index_original'})
              .assign(category = 'auxotroph')
              .astype({'category': 'category'})
              .assign(seq_to_order = lambda df_: df_['mismatch_sgrna'].apply(sgrna_subsample.add_adapter_seqs))  # Add adapter sequences
              # .loc[lambda df_: df_['seq_to_order'].apply(check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # REDUNDANT   
)

In [ ]:
# Generate the library of essential genes
df_lib_essentials = pd.concat([df_me.loc[:,['gene', 'category', 'seq_to_order']],
df_pr.loc[:,['gene', 'category', 'seq_to_order']],
df_grna_aux.loc[:,['gene', 'category', 'seq_to_order']],
df_controls.loc[:, ['category', 'seq_to_order']]])

In [ ]:
df_lib_essentials

### Nonessential genes


In [ ]:
# Get the list
# Load the reference genome
from Bio import SeqIO
headpath = '/home/gutil/scratch/2023-11-15_Mismatch-CRISPRi-Essential-Genes/'
path_to_genbank = headpath+'GCF_000009045.1_ASM904v1_genomic_Bsubtilis-168.gbff'
record = SeqIO.read(path_to_genbank, "genbank")

nonessential_genes = (pd.DataFrame([[feature.qualifiers.get('locus_tag', [''])[0],
                                     feature.qualifiers.get('gene', [''])[0]] 
                                     for feature in record.features if feature.type == 'CDS'],
                                     columns =  ['locus_tag', 'gene'])
                                # Filter out the essential and auxotroph genes
                                .loc[lambda df_: ~df_['locus_tag'].isin(
                                 df_me['locus_tag'].unique().tolist() + 
                                 df_pr['locus_tag'].unique().tolist() + 
                                #  df_controls['locus_tag'].unique().tolist() + 
                                 df_grna_aux['locus_tag'].unique().tolist())] # Remove the ~ to check for overlap
)

In [ ]:
path_to_script = '/home/gutil/lag_paulsson-lab/library_design/generate_sgrnas.py'
path_to_genbank = headpath+'GCF_000009045.1_ASM904v1_genomic_Bsubtilis-168.gbff'
path_to_params = headpath+ 'model_param_bsub.csv'
path_to_output_find = headpath+'sgrnas_nonessentials.csv'
path_to_output_filter = headpath+'sgrnas_nonessentials_filtered.csv'
path_to_output_mismatch = headpath+'sgrnas_nonessentials_mismatch.csv'

path_log = headpath+'log_2.txt'

k = 4 # Number of gRNAs per nonessential gene

In [ ]:
# Find
with open(path_log, 'w') as f:
    subprocess.run(['python', path_to_script, 
                    '--genbank', path_to_genbank,
                    '--locus_tag', ','.join(nonessential_genes['locus_tag'].to_list()),
                    '--model_param', path_to_params, 
                    '--step', 'find',
                    '--file_find', path_to_output_find], stdout=f, stderr=f)

In [ ]:
# Do not filter but evaluate the conditions above and return a series of booleans
def filter_1(df_):
    '''
    Filter the gRNAs based on the following criteria:
    - GC content < 0.6
    - PAM != 'ANN'
    - Max number of off-targets = 0
    '''
    return (
        (df_['gc'] < 0.6) & 
        (df_['pam'].str.startswith('A') == False) & 
        (df_['off_target_n'] == 0)
        )

def filter_2(df_):
    '''
    Filter the gRNAs based on the following criteria:
    - GC content < 0.7
    - PAM != 'ANN'
    - Max number of off-targets = 0
    '''
    return (
        (df_['gc'] < 0.7) & 
        (df_['pam'].str.startswith('A') == False) & 
        (df_['off_target_n'] == 0)
        )

def filter_3(df_):
    '''
    Filter the gRNAs based on the following criteria:
    - GC content < 0.6
    - PAM != 'ANN'
    - Max number of off-targets = 0
    '''
    return (
        (df_['gc'] < 0.6) & 
        (df_['pam'].str.startswith('A') == False) & 
        (df_['off_target_n'] <= 1)
        )

In [ ]:
df_noness = (pd.read_csv(path_to_output_find,
                  usecols = ['locus_tag', 'gene_name', 'strand', 'n', 'offset',
       'position_offset', 'gc', 'sgrna', 'pam', 'downstream', 'sequence',
       'seed', 'off_target_n']
      )
      .rename(columns={'gene_name': 'gene'})
      .astype({'strand': 'category',
               'n': 'int32',
               'offset': 'int32',
               'position_offset': 'int32',
               'gc': 'float64',
               'pam': 'category',
               'off_target_n': 'int32'})
      .loc[lambda df_: df_['sgrna'].duplicated(keep=False) == False] # Remove duplicate sgRNAs (both of them)
      # Check for restriction sites
      .loc[lambda df_: df_['sgrna'].apply(sgrna_subsample.check_all_sites) == False]
      .loc[lambda df_: df_['sgrna'].apply(sgrna_subsample.add_adapter_seqs).apply(sgrna_subsample.check_all_sites, args=[["GGTCTC", "CGATCG"]]) == False] # Remove rows with BsaI and PvuI sites
      .assign(pass_filter_1 = filter_1)
      .assign(pass_filter_2 = filter_2)
      .assign(pass_filter_3 = filter_3)
)

In [ ]:
df_noness

In [ ]:
df_grna_noness =(df_noness
                 .sort_values(['locus_tag','pass_filter_1', 'pass_filter_2', 'pass_filter_3'], ascending = [True, False, False, False])
                 .groupby('locus_tag')
                 .head(k)
                 .assign(seq_to_order = lambda df_: df_['sequence'].apply(sgrna_subsample.add_adapter_seqs))
                 .assign(category = 'nonessential')
                 .astype({'category': 'category'})
)

In [ ]:
df_lib_nonessentials = pd.concat([df_grna_noness.loc[:, ['gene', 'category', 'seq_to_order']],
df_controls.loc[:, ['category', 'seq_to_order']]])

In [ ]:
df_lib_nonessentials

# Sandbox

In [ ]:
# Test for a single gene subset
# gene = 'ftsZ'
# df_sample = generate_sample_single_gene(df[df['gene'] == gene],
#                                         bin_dict=bin_dict,
#                                         k=2,
#                                         col='relative fitness (mean)',
#                                         bin_index_name='bin_index_fitness')
# ax = df_sample[df_sample['gene']==gene].plot(x='predicted_efficacy', y='relative fitness (mean)', kind='scatter', title=gene, figsize=(3,3))
# df_sample.plot(x='predicted_efficacy', y='relative fitness (mean)', kind='scatter', color='red', ax=ax)
# plt.xlim([-0.1, 1.1])
# plt.ylim([-0.1, 1.1]);

In [ ]:
# Plot 
gene_name = 'murAA'
bin_colors = {0: 'red', 1: 'orange', 2: 'yellow', 3: 'green', 4: 'cyan',
              5: 'blue',6: 'magenta', 7: 'gold', 8: 'lime', 9:'teal', -1: 'black'}
# Map the bin indices to colors
# colors = df_gene['bin_index'].map(bin_colors)

# Make a 1 row x 3 column grid of plots
fig, axes = plt.subplots(1, 3, figsize=(9, 3), sharex = True, sharey=True)

df.loc[df['gene'] == gene_name].plot.scatter(
    x='predicted_efficacy',
    y='relative fitness (mean)',
    c='black',
    ax=axes[0],
    ylim = [-0.1, 1.1],
    xlim = [-0.1, 1.1],
)

df_me.loc[df_me['gene'] == gene_name].plot.scatter(
    x='predicted_efficacy',
    y='relative fitness (mean)',
    c='red',
    ax=axes[0],
    label = 'Fit'
)

df_pr.loc[df_pr['gene'] == gene_name].plot.scatter(
    x='predicted_efficacy',
    y='relative fitness (mean)',
    c='blue',
    ax=axes[0],
    label = 'KD',
    xlabel = 'Predicted KD level',
    ylabel = 'Relative fitness'
)

df_me.loc[df_me['gene'] == gene_name].plot.scatter(
    x='predicted_efficacy',
    y='relative fitness (mean)',
    c=df_me.loc[df_me['gene'] == gene_name, 'bin_index_fitness'].map(bin_colors),
    ax=axes[1],
    title = 'Binned on Fitness',
    xlabel = 'Predicted KD level',
)

df_pr.loc[df_pr['gene'] == gene_name].plot.scatter(
    x='predicted_efficacy',
    y='relative fitness (mean)',
    c=df_pr.loc[df_pr['gene'] == gene_name, 'bin_index'].map(bin_colors),
    ax=axes[2],
    title = 'Binned on KD level',
    xlabel = 'Predicted KD level',   
)

# Include vertical lines on each bin boundary
axes[1].axhline(0, color='black', linestyle='--')
axes[2].axvline(0, color='black', linestyle='--')
for i in range(len(bin_dict) - 1):
    axes[1].axhline(bin_dict[i][1], color='black', linestyle='--')
    axes[2].axvline(bin_dict[i][1], color='black', linestyle='--')

In [ ]:
# Plot for the outputs of the paper code
# gene_name = 'murAA'
locus_tag = 'BSU_40420'
bin_colors = {0: 'red', 1: 'orange', 2: 'yellow', 3: 'green', 4: 'cyan',
              5: 'blue',6: 'magenta', 7: 'gold', 8: 'lime', 9:'teal', -1: 'black'}
# Map the bin indices to colors
# colors = df_gene['bin_index'].map(bin_colors)
# df.loc[df['locus_tag'] == locus_tag, 'predicted efficacy'].hist()

# Make a swarm plot of the predicted efficiency
df.loc[df['locus_tag'] == locus_tag].plot.scatter(x='predicted efficacy', y='position')

In [ ]:
bin_colors = {0: 'red', 1: 'orange', 2: 'yellow', 3: 'green', 4: 'cyan',
              5: 'blue',6: 'magenta', 7: 'gold', 8: 'lime', 9:'teal', -1: 'black'}
df.loc[df['locus_tag'] == locus_tag].plot.scatter(x='predicted efficacy',
                                                  y='position',
                                                  c=df.loc[df['locus_tag'] == locus_tag,'bin_index'].map(bin_colors))

In [ ]:
# PLOT THE SELECTED (SAMPLED) AND ALL OF THEM TOGETHER

# # Plot the position and the predicted efficacy of the gRNAs for a given gene
# gene = 'argB'
# # locus_tag = 'BSU_40420'

# df_mm[df_mm['gene'] == gene].plot.scatter(x='predicted efficacy',
#                                             y='position',
#                                             c=df_mm.loc[df_mm['gene'] == gene,'bin_index'].map(bin_colors))

# df[df['gene'] == gene].plot.scatter(x='predicted efficacy',
#                                             y='position',
#                                             c=df.loc[df['gene'] == gene,'bin_index'].map(bin_colors))
# # Plot the position and the predicted efficacy of the gRNAs for a given gene
gene = 'argB'

# Get the data for the given gene from df_mm and df
df_mm_gene = df_me[df_me['gene'] == gene]
df_gene = df[df['gene'] == gene]

# Create a scatter plot
plt.scatter(df_mm_gene['predicted_efficacy'], df_mm_gene['position'], c=df_mm_gene['bin_index'].map(bin_colors), label='df_mm')
plt.scatter(df_gene['predicted_efficacy'],
            df_gene['position'],
            # c = 'r',
            c=df_gene['bin_index'].map(bin_colors),
            label='df',
            marker = '.')

# Add labels and legend
plt.xlabel('Predicted Efficacy')
plt.ylabel('Position')
plt.legend()

# Show the plot
plt.show()


In [ ]:
# Read Excel file
# table_url = "https://ars.els-cdn.com/content/image/1-s2.0-S2405471220303665-mmc4.xlsx"
df = (pd.read_excel(headpath + filename_essentials,
                    sheet_name="relative fitness (bsu data)")
    .query("family_retained == True") # Remove aberrant gRNAs according to the paper
    .drop(columns=["family_retained"]) # Drop column
    .rename(columns={'relative fitness (predicted)':'predicted_efficacy',
                     'relative fitness (mean)':'relative_fitness_mean',
                     'relative fitness (stddev)':'relative_fitness_std',
                     'variant': 'mismatch_sgrna',
                     'original': 'full_sgrna'}) # Rename column
    .astype({"pam": "category"}) # Set type of pam to categorical
    .query("pam != 'AGG'") # Only consider non-AGG PAMs
    .query("offset <= 1000") # Offset less than 1,000
    .sort_values(["gene", "relative_fitness_mean"])
    # Remove row with nans
    .dropna(subset=['relative_fitness_mean'])
    .assign(locus_tag = lambda df_: df_['locus_tag'].str.slice(0, 3)+'_'+df_['locus_tag'].str.slice(3)) # Add underscore to locus tag
    # Assign bin indices
    .assign(bin_index_fitness = lambda df_:sgrna_subsample.get_bin_ids(df_,
                                                                    col='relative_fitness_mean',
                                                                    bin_dict=bin_dict))
    .query("bin_index_fitness != -1")
 
    .assign(bin_index = lambda df_:sgrna_subsample.get_bin_ids(df_,
                                                               col='predicted_efficacy',
                                                               bin_dict=bin_dict))
    .query("bin_index != -1")
    # Select the rows whose mismatch_grna does not contain any restriction site
    .assign(has_restriction_sites = lambda df_: df_['mismatch_sgrna'].apply(check_all_sites))
    .query("has_restriction_sites == False")
)
df

In [ ]:
# Get the first k gRNAs for each gene
df_grna_filter_0 = (df_noness_filter_0
                     .groupby('gene')
                     .head(k) # Get the first k gRNAs for each gene
)

# Remove genes from noness given a list of locus_tags
df_grna_filter_0['locus_tag'].value_counts() == k

# Remove locus_tags of genes for which k gRNAs have been selected
locus_tags_to_remove = (df_grna_filter_0['locus_tag']
                        .value_counts()
                        .loc[lambda s_: s_ == k]
                        .index
                        .to_list()
)

# Remove those from the unfiltered df
df_noness = (df_noness.loc[~df_noness['locus_tag'].isin(locus_tags_to_remove)]
)

# Remove entries from df_noness with the same locus_tag and n as the ones in df_grna_filter_0
df_noness = (df_noness
             .loc[~df_noness.set_index(['locus_tag', 'n']).index.isin(df_grna_filter_0.set_index(['locus_tag', 'n']).index)]
)